In [ ]:
#use "../lib/nb.ml";;
run_notebook "Types.ipynb"

# Checking the Equivalence of Regular Expressions

In order to check whether two regular expressions $r_1$ and $r_2$ are *equivalent*, perform the 
following steps:
- convert the regular expressions $r_1$ and $r_2$ into *NFAs*
  $F_1$ and $F_2$ such that $L(r_1) = L(F_1)$ and $L(r_2) = L(F_2)$,
- convert the *NFAs* $F_1$ and $F_2$ into *DFAs*
  $D_1$ and $D_2$ such that $L(D_1) = L(F_1)$ and $L(D_2) = L(F_2)$
- check whether both $L(D_1) \backslash L(D_2)$ and $L(D_2) \backslash L(D_1)$ are empty.

The notebook `03-Regexp-2-NFA.ipynb` contains the function `toNFA` that can be used to compute an
<span style="font-variant:small-caps;">Nfa</span> that accepts the language described by a given regular expression.

In [ ]:
run_notebook "03-Regexp-2-NFA.ipynb"

The notebook `01-NFA-2-DFA.ipynb` contains the function `nfa2dfa` that converts an <span style="font-variant:small-caps;">Nfa</span> into an equivalent <span style="font-variant:small-caps;">Dfa</span>.

In [ ]:
run_notebook "01-NFA-2-DFA.ipynb"

Given two sets `A` and `B`, the function `cartesian_product a b` computes the 
<em style="color:blue">cartesian product</em> $A \times B$ which is defined as
$$ A \times B := \{ (x, y) \mid x \in A \wedge y \in B \}. $$

**Function `cartesian_product a b`**
- *Input:* `a` and `b` are sets.
- *Output:* The Cartesian product $A \times B$.

In [ ]:
let cartesian_product (a : 's FSet.t) (b : 't FSet.t) : ('s * 't) FSet.t =
  let open FSet in
  let* x = a in
  let* y = b in
  return (x, y)

In [ ]:
cartesian_product (FSet.of_list [ 1; 2 ]) (FSet.of_list [ 'a'; 'b' ])

In Python, the types `DFA1` and `DFA2` describe the two kinds of finite state machines used in this notebook:
`DFA1` has states of an arbitrary type `State`, while the states of `DFA2` are pairs of states.  In OCaml, these
types are `'p dfa` and `('p * 'q) dfa`.  Note that the two machines combined by `fsm_complement` may have states
of *different* types `'p` and `'q`.

In [ ]:
#show_type dfa

Given two <span style="font-variant:small-caps;">Dfa</span>s `F1` and `F2`, the expression `fsm_complement f1 f2` computes a <span style="font-variant:small-caps;">Dfa</span>
that recognizes the language  $L(F_1)\backslash L(F_2)$.

**Function `fsm_complement f1 f2`**
- *Input:* `f1` and `f2` are complete <span style="font-variant:small-caps;">Dfa</span>s with the same alphabet.
- *Output:* A <span style="font-variant:small-caps;">Dfa</span> that accepts the language $L(F_1) \backslash L(F_2)$.

The transition function is built in a loop.  Since dictionaries are immutable, `delta` is a reference to a
dictionary.

In [ ]:
let fsm_complement (f1 : 'p dfa) (f2 : 'q dfa) : ('p * 'q) dfa =
  let ({ states = states1; sigma; delta = delta1; start = q1; accepting = a1 } : 'p dfa) = f1 in
  let ({ states = states2; delta = delta2; start = q2; accepting = a2; _ } : 'q dfa) = f2 in
  let states = cartesian_product states1 states2 in
  let delta = ref Dict.empty in
  FSet.iter
    (fun (p1, p2) ->
      FSet.iter (fun c -> delta := Dict.add ((p1, p2), c) (Dict.find (p1, c) delta1, Dict.find (p2, c) delta2) !delta) sigma)
    states;
  { states; sigma; delta = !delta; start = (q1, q2); accepting = cartesian_product a1 (FSet.diff states2 a2) }

In [ ]:
#show_type regexp

Given a regular expression $r$ and an alphabet $\Sigma$, the function $\texttt{regexp2DFA}(r, \Sigma)$
computes a *DFA* that accepts the language specified by $r$.

The Python version has to use `# type: ignore` here because the class `RegExp2NFA` and the function `nfa2dfa`
have been defined in a different notebook.  In OCaml, `run_notebook` makes these definitions known to the
compiler, so the function is type checked completely.

**Function `regexp2DFA r sigma`**
- *Input:* `r` is a regular expression and `sigma` is an alphabet.
- *Output:* A <span style="font-variant:small-caps;">Dfa</span> that accepts the language described by `r`.

In [ ]:
let regexp2DFA (r : regexp) (sigma : char FSet.t) : state FSet.t dfa =
  let converter = regExp2NFA sigma in
  let nfa       = toNFA converter r in
  nfa2dfa nfa

Given a <span style="font-variant:small-caps;">Dfa</span> $F$ the function 
`is_empty f` checks whether the language accepted by $F$ is empty.
In this function, the variable `reachable` is the set of those states that are already known to be reachable
from the start state `q0`. `newFound` are those states that can be reached from a state in the set 
`reachable`.  When we find no new states that are reachable, the iteration stops and we check whether
there is a state that is both reachable and accepting because in that case the language is not empty.

**Function `is_empty f`**
- *Input:* `f` is a <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* `true` if $L(F)$ is empty, `false` otherwise.

In [ ]:
let is_empty (f : 'q dfa) : bool =
  let ({ sigma; delta; start = q0; accepting; _ } : 'q dfa) = f in
  let rec loop (reachable : 'q FSet.t) : 'q FSet.t =
    let newFound =
      let open FSet in
      let* q = reachable in
      let* c = sigma in
      return (Dict.find (q, c) delta)
    in
    if FSet.subset newFound reachable then reachable
    else loop (FSet.union reachable newFound)
  in
  FSet.is_empty (FSet.inter (loop (FSet.singleton q0)) accepting)

The function `regExpEquiv` takes three arguments:
- $r_1$ and $r_2$ are regular expressions,
- $\Sigma$ is the alphabet used in these regular expressions.

The function returns `true` iff $r_1 \doteq r_2$, i.e. if $r_1$ and $r_2$ are equivalent. 

**Function `regExpEquiv r1 r2 sigma`**
- *Input:* `r1` and `r2` are regular expressions and `sigma` is the alphabet used in them.
- *Output:* `true` if $r_1 \doteq r_2$, `false` otherwise.

In [ ]:
let regExpEquiv (r1 : regexp) (r2 : regexp) (sigma : char FSet.t) : bool =
  let f1 = regexp2DFA r1 sigma in
  let f2 = regexp2DFA r2 sigma in
  let r1_minus_r2 = fsm_complement f1 f2 in
  let r2_minus_r1 = fsm_complement f2 f1 in
  is_empty r1_minus_r2 && is_empty r2_minus_r1

The notebook `10-Test-Equivalence.ipynb` can be used to test this function.